In [1]:
# pip install langchain langchain-google-genai langchain-classic python-dotenv

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import os
if not os.getenv("GEMINI_API_KEY"):
    print("Gemini API key is not available.")
else:
    print("Gemini API key is available.")

Gemini API key is available.


In [5]:
# LIST OF GEMINI MODEL TO TRY IF ONE MODEL FAILS
MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"]

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

def get_model():
    # TRY EACH MODEL ONE BY ONE
    for name in MODELS:
        try:
            # CREATE A GEMINI MODEL
            llm = ChatGoogleGenerativeAI(model=name, temperature=0.3)

            # TEST THE MODEL
            llm.invoke("Say OK")

            # RETURN THE WORKING MODEL
            print("Using model:", name)
            return llm

        except:
            # IF THE CURRENT MODEL FAILS, TRY THE NEXT MODEL
            print(name, "failed")

    # RETURN THIS MESSAGE IF ALL MODELS FAIL
    return "NO GEMINI MODEL WORKS CHECK WITH THE API KEY AND MODEL NAMES"

In [7]:
# FUNCTION TO GET THE TEXT FROM THE MODEL RESPONSE
def text_of(message):
    return str(message.content)

In [8]:
from langchain_core.prompts import ChatPromptTemplate

# INITIALISING MODEL
model = get_model()
prompt = ChatPromptTemplate.from_template("Explain {topic} in one sentence as if you are talking to a beginner.")

# CONNECTING THE PROMPT AND MODEL INTO CHAIN USING LCEL PIPE OPERATOR (|)
chain = prompt | model

# SEND(INVOKE) THE TOPIC THROUGH THE CHAIN
response = chain.invoke({"topic": "Quantum Computing"})

# PRINT THE MODEL RESPONSE
print(text_of(response))

Using model: gemini-3.5-flash-lite
[{'type': 'text', 'text': 'While regular computers think in strict 0s and 1s, quantum computers use magical spinning coin-like bits that can be 0 and 1 at the same time, allowing them to solve massive problems instantly.', 'extras': {'signature': 'EpYBCpMBAWkUfRNkmsCarDSTr568nuGxkzCbQdca/xVyqiYhbB0KHrOgCUSwOpSrGzeN/A+Jaz+g8q5kvjtuHey1lKHH41lgXHX5BQCpao15WMiad0dP4iOMAlRqCtEbIDv+OHMIpDjl36Q5JyOkHl5FqdqkQITn9q2YBqCY5Xpz5ny5otYqS0IbgazwfjTtPQR/566cV9qQ'}}]


In [9]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

# CREATE THE OUTPUT PARSER
parser = CommaSeparatedListOutputParser()

# CREATE A PROMPT WITH FORMATTING INSTRUCTIONS
prompt2 = PromptTemplate(
    template="Explain {topic} in simple words and provide three key points.\n{format_instructions}",
    input_variables=["topic"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# CREATE THE OUTPUT PARSER
parser = CommaSeparatedListOutputParser()

# CREATE A PROMPT WITH FORMATTING INSTRUCTIONS
prompt2 = PromptTemplate(
    template=("Give a simple explanation and three key points about {topic}.\n"
              "Return exactly 4 items. The first item should be a simple explanation "
              "in maximum 12 words, followed by three short key points. "
              "Do not use commas inside any item.\n"
              "{format_instructions}"),
    input_variables=["topic"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
# COMMAS ARE NOT USED WITHIN INDIVIDUAL ITEMS BECUASE THE CommaSeparatedListOutputParser USES COMMAS TO SEPARATE THE INSTRUCTION.

# CONNECTING THE PROMPT, MODEL AND PARSER INTO CHAIN USING LCEL PIPE OPERATOR (|)
chain2 = prompt2 | model | parser

# SEND(INVOKE) THE TOPIC THROUGH THE CHAIN
result = chain2.invoke({"topic": "Machine Learning"})

# PRINT THE COMPLETE RESULT
print("Parsed list:", result)

# PRINT THE EXPLANATION
print("\nExplanation:", result[0])

# PRINT THE THREE KEY POINTS
print("Key points:")
for i, point in enumerate(result[1:], start=1):
    print(f"{i}. {point}")

Parsed list: ['Computers learning from data without explicit programming', 'Algorithms find patterns in data', 'Models improve with experience', 'Predictions made on new data']

Explanation: Computers learning from data without explicit programming
Key points:
1. Algorithms find patterns in data
2. Models improve with experience
3. Predictions made on new data


In [11]:
# ConversationBufferMemory now lives in langchain_classic (older versions: langchain.memory)
# IMPORT CONVERSATION BUFFER MEMORY
try:
    from langchain_classic.memory import ConversationBufferMemory
except ImportError:
    from langchain.memory import ConversationBufferMemory


# CREATE THE MEMORY BUFFER
memory = ConversationBufferMemory(memory_key="conversation_history", return_messages=False)    

C:\Users\excel\AppData\Local\Temp\ipykernel_7468\4004323775.py:10: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="conversation_history", return_messages=False)
c:\Users\excel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pydantic\main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.valid

In [12]:
# CREATE A PROMPT WITH CONVERSATION HISTORY AND USER INPUT
prompt3 = ChatPromptTemplate.from_messages([("system",
                                             "You are a friendly Study Buddy. Explain things simply and remember "
                                             "what the student tells you.\n\n"
                                             "Conversation so far:\n{conversation_history}"),
                                             ("human", "{input}")])

# CONNECTING THE PROMPT AND MODEL INTO CHAIN USING LCEL PIPE OPERATOR (|)
chain3 = prompt3 | model

# CREATE A FUNCTION TO CHAT WITH THE STUDY BUDDY (LOAD,INVOKE,SAVE)
def chat(user_input):

    # LOAD PREVIOUS CONVERSATION HISTORY
    history = memory.load_memory_variables({})["conversation_history"]

    # SEND USER INPUT AND PREVIOUS HISTORY TO THE MODEL
    response = chain3.invoke({"input": user_input,"conversation_history": history})

    # EXTRACT THE RESPONSE TEXT
    answer = text_of(response)

    # SAVE THE USER INPUT AND MODEL RESPONSE IN MEMORY
    memory.save_context({"input": user_input},{"output": answer})

    # RETURN THE MODEL RESPONSE
    return answer

In [13]:
# SIMULATE A SHORT CONVERSATION
print("You: Hi! My name is Nagesh.")
print("Study Buddy:", chat("Hi! My name is Nagesh."))

print("\nYou: Can you explain what a neural network is in two sentences?")
print("Study Buddy:", chat("Can you explain what a neural network is in two sentences?"))

print("\nYou: Do you remember my name?")
print("Study Buddy:", chat("Do you remember my name?"))

print("\nYou: What is the meaning of my name?")
print("Study Buddy:", chat("What is the meaning of my name?"))

print("\nYou: What is the meaning of name Saishwar?")
print("Study Buddy:", chat("What is the meaning of name Saishwar?"))

You: Hi! My name is Nagesh.
Study Buddy: [{'type': 'text', 'text': "Hi Nagesh! It's so nice to meet you. I'm your Study Buddy, and I'm here to help make learning fun and easy. \n\nWhat are we working on today? Is there a specific subject or topic on your mind?", 'extras': {'signature': 'EpYBCpMBAWkUfRN/iUQIW9o+EpCj+Rr1lx9EHN7lGWnLKNVz+2m75aWPxc+RwKU6RCqHEsRRH7boA+w7fk9zSKNgeZXRw5Z8w4/sB+Ll+ojDf5dmW4S4+AENG+og/zdmPHqmtFd+kfEYskEJf4wgowO5qKYBDeYuUaCdSVrIfvtu8cBGM7nHH3ZADnW1jtHDTc5DQVcKQ9BT'}}]

You: Can you explain what a neural network is in two sentences?
Study Buddy: [{'type': 'text', 'text': 'Hey Nagesh! A neural network is a computer system inspired by the human brain that learns to recognize patterns from lots of data. It uses layers of interconnected "nodes" to process information and make decisions, just like we do when we learn new things!', 'extras': {'signature': 'EpYBCpMBAWkUfRNq5vPuWz3cuqUtMYz8waBBOMt6WTUP99hhEWUtVY7P3yBdMSnm9Ob/rz4eQ6ppGFPV65FIpKgiPtlJYvuNYvLHaWOR1hjl0NDDon

In [14]:
# DISPLAY THE STORED CONVERSATION HISTORY
print(memory.load_memory_variables({})["conversation_history"])

Human: Hi! My name is Nagesh.
AI: [{'type': 'text', 'text': "Hi Nagesh! It's so nice to meet you. I'm your Study Buddy, and I'm here to help make learning fun and easy. \n\nWhat are we working on today? Is there a specific subject or topic on your mind?", 'extras': {'signature': 'EpYBCpMBAWkUfRN/iUQIW9o+EpCj+Rr1lx9EHN7lGWnLKNVz+2m75aWPxc+RwKU6RCqHEsRRH7boA+w7fk9zSKNgeZXRw5Z8w4/sB+Ll+ojDf5dmW4S4+AENG+og/zdmPHqmtFd+kfEYskEJf4wgowO5qKYBDeYuUaCdSVrIfvtu8cBGM7nHH3ZADnW1jtHDTc5DQVcKQ9BT'}}]
Human: Can you explain what a neural network is in two sentences?
AI: [{'type': 'text', 'text': 'Hey Nagesh! A neural network is a computer system inspired by the human brain that learns to recognize patterns from lots of data. It uses layers of interconnected "nodes" to process information and make decisions, just like we do when we learn new things!', 'extras': {'signature': 'EpYBCpMBAWkUfRNq5vPuWz3cuqUtMYz8waBBOMt6WTUP99hhEWUtVY7P3yBdMSnm9Ob/rz4eQ6ppGFPV65FIpKgiPtlJYvuNYvLHaWOR1hjl0NDDonaRbZerENeiRBNXq

<hr>

<h3 style="text-align:center; color:green;">
    End of Assignment
</h3>

<hr>